# 02 — Vérifier la connexion à Assetto Corsa

Diagnostic TCP **lecture seule** : vérifie la session de jeu et la configuration de `sensors_par`. Lance Assetto Corsa en piste.

In [1]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaEnv').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvre le depot IA_projet_Ouarti_ds_github dans VS Code : son dossier assetto_corsa_gym doit exister.')
PROJECT=REPO/'Code'
if not PROJECT.is_dir():
    raise FileNotFoundError('Dossier Code absent : extraire Code a la racine du depot.')
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
print('Depot :',REPO)
print('Notebooks :',PROJECT)
print('Python :',sys.version.split()[0],sys.executable)


Depot : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github
Notebooks : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code
Python : 3.9.13 c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe


In [2]:
import socket, json, ast, time
HOST='127.0.0.1'
PORT_UDP=2345
PORT_TCP=2347

def management(message,timeout=3):
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout);s.sendall(message.encode('ascii'))
        data=b''
        end=time.monotonic()+timeout
        while time.monotonic()<end and len(data)<1024*1024:
            try:chunk=s.recv(65536)
            except socket.timeout:break
            if not chunk:break
            data+=chunk
            try:return json.loads(data)
            except (json.JSONDecodeError, UnicodeDecodeError):pass
            try:return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeDecodeError):pass
        if data:
            raise RuntimeError('Reponse TCP incomplete : '+repr(data[:160]))
    raise RuntimeError('Pas de reponse sur le port management')

def open_telemetry(timeout=4):
    s=socket.socket(socket.AF_INET,socket.SOCK_DGRAM);s.settimeout(.4)
    s.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        s.send(b'connect')
        try:response=s.recv(2**18)
        except socket.timeout:continue
        if response==b'identified':return s
    s.close()
    raise TimeoutError('Pas de confirmation UDP sur 2345 : verifier jeu en piste et plugin sensors_par.')

def next_packet(s,timeout=3):
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        try:message=s.recv(2**18)
        except socket.timeout:continue
        if message==b'disconnect':raise ConnectionError('Le jeu a ferme le flux UDP')
        if message==b'identified':continue
        raw=json.loads(message)
        if not isinstance(raw,dict):continue
        raw['_capture_wall_time']=time.time()
        return raw
    raise TimeoutError('Aucun echantillon recu.')

def close_telemetry(s):
    if s is not None:
        try:s.send(b'disconnect')
        except OSError:pass
        s.close()


In [3]:
config=management('get_config')
static=management('get_static_info')
print('Plugin :')
for key in ['ego_sampling_freq','sampling_freq','vjoy_executed_by_server','screen_capture_enable','color_mode','final_image_width','final_image_height']:
    print(' ',key,':',config.get(key))
print('Session :',static.get('TrackName'),static.get('TrackConfiguration'), '/', static.get('CarName'))


Plugin :
  ego_sampling_freq : 25
  sampling_freq : 50
  vjoy_executed_by_server : False
  screen_capture_enable : False
  color_mode : gray
  final_image_width : 320
  final_image_height : 240
Session : ks_barcelona layout_gp / abarth500
